<a href="https://colab.research.google.com/github/antonDinkov/AI_integrationsForDev/blob/main/exerciseOpenAi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PREPARATION

In [2]:
!pip install -q openai

In [3]:
!pip install -q anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 13.2 MB/s eta 0:00:00


In [12]:
from openai import OpenAI
from anthropic import Anthropic
from google.colab import userdata
from pydantic import BaseModel, Field
from pprint import pprint
from pathlib import Path

openai_key = userdata.get('OPEN_AI_API_KEY')
anthropic_key = userdata.get('ANTHROPIC_API_KEY')

openai_client = OpenAI(api_key=openai_key)
anthropic_client = Anthropic(api_key=anthropic_key)

def get_anthropic_message_lines(message):
    return [x.text for x in message.content if x.type == 'text']


def print_anthropic_message(message):
    print(f"Message id: {message.id}")
    print(f"Input: {message.usage.input_tokens}; Output: {message.usage.output_tokens}")
    print(f"Stop reason: {message.stop_reason}")
    pprint(message.content)

    print()
    thinking_content_elements = [x.thinking for x in message.content if x.type == 'thinking']
    if len(thinking_content_elements) > 0:
        print(f"{'-' * 20} [Thinking] {'-' * 20}")
        for line in thinking_content_elements:
            print(line)

    print()
    print(f"{'-' * 20} [Text] {'-' * 20}")
    for line in get_anthropic_message_lines(message):
        print(line)

def print_openai_response(response):
    print(f"Response id: {response.id}")
    print(f"Input tokens: {response.usage.input_tokens} ({response.usage.input_tokens_details.cached_tokens} cached); Output tokens: {response.usage.output_tokens} ({response.usage.output_tokens_details.reasoning_tokens} reasoning)")
    pprint(response.output)

    print()
    print(f"{'-' * 20} [Text] {'-' * 20}")
    print(response.output_text)

# Upload to Anthropic

In [6]:
path_to_file = Path("/content/test_doc_01.pdf")

with path_to_file.open("rb") as file_content:
  file_upload_response = anthropic_client.beta.files.upload(
      file=(path_to_file.name, file_content, "application/pdf")
  )

In [7]:
pprint(file_upload_response)

FileMetadata(id='file_011CckT1bTLEgLK4CabxXqdz', created_at=datetime.datetime(2026, 7, 6, 7, 58, 40, 798000, tzinfo=datetime.timezone.utc), filename='test_doc_01.pdf', mime_type='application/pdf', size_bytes=52939, type='file', downloadable=False, scope=None)


## Summarize the document

In [14]:
system_prompt = """
You are an expert summarizer of legal and technical documents.

Your task:
- Produce a concise summary of approximately 200 words.
- The summary must preserve the meaning and structure of the source text.

Hard constraints:
- Do NOT invent, assume, or infer any information not explicitly stated in the source document.
- Every statement in the summary must be directly supported by the source text.
- Do NOT add external knowledge, interpretation, or explanations.
- Do NOT introduce new entities, facts, dates, or numbers not present in the input.

Content rules:
- Preserve factual accuracy and neutrality.
- Merge and compress repeated ideas only if they are semantically identical in the source.
- Keep legal/technical meaning intact (do not simplify in a way that changes meaning).
- Remove redundancy, filler, and verbose phrasing.

Output rules:
- Target length: ~200 words (acceptable range: 180–220 words).
- Maintain formal, objective tone.

Priority rule:
- If there is any conflict between conciseness and faithfulness to the source text, faithfulness ALWAYS takes priority.
"""

file_id = file_upload_response.id

summery_response = anthropic_client.beta.messages.create(
    model="claude-haiku-4-5-20251001",
    messages=[
        {"role": "user", "content": [
            {"type": "text", "text": "Summarize the referenced document."},
            {"type": "document", "source": {
                "file_id": file_id, "type": "file"
            }}
        ]}
    ],
    max_tokens=1024,
    system=system_prompt,
    betas=["files-api-2025-04-14"]
)

In [15]:
summery_response

BetaMessage(id='msg_01Cm17yQoQqajcSS9PJjHzNv', container=None, content=[BetaTextBlock(citations=None, text='# Invoice Summary\n\n**Document:** Invoice No. 2026-00147 issued by Try at Software ООД (Shumen, Bulgaria) on 21.02.2026 to ДигиМаркет ЕООД (Plovdiv).\n\n**Issuer Details:** Try at Software ООД, EIK: 207654321, VAT ID: BG207654321, Phone: +359 2 987 6543\n\n**Recipient Details:** ДигиМаркет ЕООД, Boulevard Vitosha 128, Plovdiv 4000, EIK: 305123456, VAT ID: BG305123456\n\n**Services Rendered:**\n1. Web design and corporate website development (4,500.00 BGN)\n2. SEO optimization—initial package (3 months) (1,200.00 BGN)\n3. Premium hosting plan—annual (480.00 BGN)\n4. Graphic design—logo and brand identity (1,800.00 BGN)\n5. Technical support—monthly subscription (3 months at 350.00 BGN each = 1,050.00 BGN)\n6. Wildcard SSL certificate—annual (320.00 BGN)\n7. Payment system integration (Stripe) (750.00 BGN)\n\n**Financial Summary:**\n- Tax Base: 10,100.00 BGN\n- VAT (20%): 2,020.00

In [16]:
print_anthropic_message(summery_response)

Message id: msg_01Cm17yQoQqajcSS9PJjHzNv
Input: 2525; Output: 429
Stop reason: end_turn
[BetaTextBlock(citations=None, text='# Invoice Summary\n\n**Document:** Invoice No. 2026-00147 issued by Try at Software ООД (Shumen, Bulgaria) on 21.02.2026 to ДигиМаркет ЕООД (Plovdiv).\n\n**Issuer Details:** Try at Software ООД, EIK: 207654321, VAT ID: BG207654321, Phone: +359 2 987 6543\n\n**Recipient Details:** ДигиМаркет ЕООД, Boulevard Vitosha 128, Plovdiv 4000, EIK: 305123456, VAT ID: BG305123456\n\n**Services Rendered:**\n1. Web design and corporate website development (4,500.00 BGN)\n2. SEO optimization—initial package (3 months) (1,200.00 BGN)\n3. Premium hosting plan—annual (480.00 BGN)\n4. Graphic design—logo and brand identity (1,800.00 BGN)\n5. Technical support—monthly subscription (3 months at 350.00 BGN each = 1,050.00 BGN)\n6. Wildcard SSL certificate—annual (320.00 BGN)\n7. Payment system integration (Stripe) (750.00 BGN)\n\n**Financial Summary:**\n- Tax Base: 10,100.00 BGN\n- VA

In [20]:
document_summary = "\n".join(get_anthropic_message_lines(summery_response))

In [21]:
document_summary

'# Invoice Summary\n\n**Document:** Invoice No. 2026-00147 issued by Try at Software ООД (Shumen, Bulgaria) on 21.02.2026 to ДигиМаркет ЕООД (Plovdiv).\n\n**Issuer Details:** Try at Software ООД, EIK: 207654321, VAT ID: BG207654321, Phone: +359 2 987 6543\n\n**Recipient Details:** ДигиМаркет ЕООД, Boulevard Vitosha 128, Plovdiv 4000, EIK: 305123456, VAT ID: BG305123456\n\n**Services Rendered:**\n1. Web design and corporate website development (4,500.00 BGN)\n2. SEO optimization—initial package (3 months) (1,200.00 BGN)\n3. Premium hosting plan—annual (480.00 BGN)\n4. Graphic design—logo and brand identity (1,800.00 BGN)\n5. Technical support—monthly subscription (3 months at 350.00 BGN each = 1,050.00 BGN)\n6. Wildcard SSL certificate—annual (320.00 BGN)\n7. Payment system integration (Stripe) (750.00 BGN)\n\n**Financial Summary:**\n- Tax Base: 10,100.00 BGN\n- VAT (20%): 2,020.00 BGN\n- **Total Amount Due: 12,120.00 BGN**\n\n**Payment Details:** ProBank AD, IBAN: BG42PROB9100103254789

In [22]:
delete_file = anthropic_client.beta.files.delete(
    file_id
)

In [23]:
delete_file

DeletedFile(id='file_011CckT1bTLEgLK4CabxXqdz', type='file_deleted')